In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

df = pd.read_csv("../data/processed/labeled_prs_v2.csv")

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["title"].fillna("").tolist(), show_progress_bar=True)

numeric = ["additions", "deletions", "changed_files", "commits",
           "comments", "review_comments", "num_files",
           "author_past_prs", "author_past_bug_rate", "is_first_pr"]

emb_small = PCA(n_components=30, random_state=42).fit_transform(embeddings)
X = np.hstack([df[numeric].values, emb_small])
y = df["label"].values
print(X.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

(500, 40)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier 
from lightgbm import LGBMClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                             random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                               random_state=42, verbose=-1),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    results[name] = auc
    print(f"\n=== {name} ===")
    print(classification_report(y_test, m.predict(X_test)))
    print("AUC:", round(auc, 3))



=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.69      0.90      0.78        60
           1       0.73      0.40      0.52        40

    accuracy                           0.70       100
   macro avg       0.71      0.65      0.65       100
weighted avg       0.71      0.70      0.68       100

AUC: 0.662

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.68      0.83      0.75        60
           1       0.62      0.40      0.48        40

    accuracy                           0.66       100
   macro avg       0.65      0.62      0.62       100
weighted avg       0.65      0.66      0.64       100

AUC: 0.71

=== XGBoost ===
              precision    recall  f1-score   support

           0       0.66      0.75      0.70        60
           1       0.53      0.42      0.47        40

    accuracy                           0.62       100
   macro avg       0.60      0.59     

c:\Users\Hp\AutoDebugAI\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Hp\AutoDebugAI\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
leaderboard = pd.Series(results).sort_values(ascending=False)
print(leaderboard)

Random Forest          0.710417
Logistic Regression    0.661667
XGBoost                0.657917
LightGBM               0.630833
dtype: float64


In [6]:
import joblib
from pathlib import Path

winner_name = leaderboard.index[0]
winner = models[winner_name]
Path("../models").mkdir(exist_ok=True)
joblib.dump(winner, "../models/best_model.pkl")
print(f"Saved {winner_name} (AUC {leaderboard.iloc[0]:.3f}) to models/best_model.pkl")

Saved Random Forest (AUC 0.710) to models/best_model.pkl


In [7]:
import joblib

model = joblib.load("../models/best_model.pkl")
print(model)                      # RandomForestClassifier(n_estimators=200, ...)
print(model.n_features_in_)       # 40 — expects your feature set
print(model.classes_)             # [0 1]

RandomForestClassifier(n_estimators=200, random_state=42)
40
[0 1]


In [ ]:
import numpy as np
fake_pr = np.random.rand(1, 40)   # one row, 40 features
print(model.predict_proba(fake_pr))   


[[0.57 0.43]]
